<a href="https://colab.research.google.com/github/IzanPereira/Projetos-de-estudos/blob/main/An%C3%A1lise_de_sentimentos_com_biblioteca_pronta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import polars as pl
!pip install pysentimiento
from pysentimiento import create_analyzer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import plotly.express as px

In [ ]:
# Instanciamos o analisador focado no idioma Português
# Esse modelo já foi treinado com milhões de tweets e textos da internet
analyzer = create_analyzer(task="sentiment", lang="pt")

# Carregamos o seu dataset
# Substitua pelo caminho real do seu arquivo
df = pl.read_csv('comments.csv')

# Isolamos apenas a coluna de texto e removemos valores nulos
df_textos = df.select(['textOriginal']).drop_nulls()

Esse o modelo original

In [53]:
# Função auxiliar para extrair apenas a classe (POS, NEG, NEU)
#def prever_sentimento(texto):
    #try:
        # O modelo processa o texto e retorna um objeto detalhado
       # resultado = analyzer.predict(texto)
        # Extraímos apenas a string com a classe vencedora
        #return resultado.output
    #except Exception as e:
        #return "ERRO"

# Aplicamos a função linha a linha utilizando a velocidade do Polars
#print("Iniciando a classificação dos comentários...")
#df_resultado = df_textos.with_columns(
    #pl.col("textOriginal")
    #.map_elements(prever_sentimento, return_dtype=pl.Utf8)
    #.alias("sentimento_predito"))
   # 🎯 Acurácia Geral do pysentimiento: 54.96%
#📊 F1-Score: 0.5178
#🔍 Matriz de Confusão:
#Predito_POS | Predito_NEG | Predito_NEU
#Real_POS: 114 | 10 | 83
#Real_NEG: 18 | 28 | 123
#Real_NEU: 30 | 22 | 207


Como base no modelo original, recalibramos para que penalize a mais, a classificação dos neutros, forçando o modelo a escolher POS ou NEG

In [ ]:
# Com essa nova função de Calibragem de Confiança
def prever_sentimento_calibrado(texto):
    try:
        # O modelo irá processa o texto e retorna o objeto detalhado
        resultado = analyzer.predict(texto)

        # Interfirimos no dicionário de probabilidades
        # recalibrando como por exemplo do que vem aqui:
        #{'POS': 0.15, 'NEG': 0.25, 'NEU': 0.60}
        probas = resultado.probas

        # Definimos o nosso sarrafo para a neutralidade (70% de certeza)
        limite_neu = 0.70

        # Regra de negócio:
        if probas['NEU'] >= limite_neu:
            return "NEU"
        else:
            # Se a certeza de Neutro for menor que 70%, forçamos a decisão
            if probas['POS'] > probas['NEG']:
                return "POS"
            else:
                return "NEG"
    except Exception as e:
        return "ERRO"

print("Iniciando a reclassificação com calibragem de confiança...")

# Aplicando a função
df_resultado = df_textos.with_columns(
    pl.col("textOriginal")
    .map_elements(prever_sentimento_calibrado, return_dtype=pl.Utf8)
    .alias("sentimento_predito")
)

In [ ]:
# Visualizamos o resultado final lado a lado
print(df_resultado.head(10))

Se quiser salvar o resultado em um novo CSV para auditar:

In [84]:
# df_resultado.write_csv("resultados_voce_sabia.csv")

Visualização Gráfica

Aqui ele vai agrupar as classificações em POS, NEG e NEU.

In [85]:
df_contagem = df_resultado.group_by("sentimento_predito").len().to_pandas()

# Mapea as cores para ficar visualmente lógico e fácil de identificar
cores = {'POS': '#Strong Cyan', 'NEG': '#Burnt Orange', 'NEU': '#Soft Blue'}
#cores = {'POS': '#00CC96', 'NEG': '#EF553B', 'NEU': '#636EFA'} esse #CODIGO é a identificação da cor em código hexadecimal de cor que pode ser usado
# Criando o gráfico de pizza
fig = px.pie(
    df_contagem,
    values='len',
    names='sentimento_predito',
    title='Distribuição de Sentimentos dos Comentários - Você Sabia',
    color='sentimento_predito',
    color_discrete_map=cores
)

# Melhorando o layout para o nosso dashboard do DataLab
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

Paneil de cores:
https://color.adobe.com/br/explore?theme=color-palettes&q=Ver%C3%A3o

AUDITORIA DO MODELO: COMPARANDO O PYSENTIMIENTO COM O SEU GABARITO

Agora avaliamos se o modelo que usamos esta com uma boa precisão, para isso precisamos garantir que o Dataframe que classificamos manualmente terá a coluna do gabarito.
Supondo que a coluna original se chame 'Sentimentos_manual' (POS, NEG, NEU) e a do modelo se chama 'sentimento_predito'

In [86]:
df_preenchido = pl.read_csv('/content/comentarios_canal_voce_sabia_preenchido1.csv', encoding='latin1', truncate_ragged_lines=True, separator=';')

In [87]:
print(df_preenchido.head(10))

shape: (10, 3)
┌────────┬─────────────────────────────────┬────────────────────┐
│ Índice ┆ Comentário Original             ┆ Sentimentos_manual │
│ ---    ┆ ---                             ┆ ---                │
│ i64    ┆ str                             ┆ str                │
╞════════╪═════════════════════════════════╪════════════════════╡
│ 1      ┆ ??                              ┆ POS                │
│ 2      ┆ O Bob esponja nasceu no mesmo … ┆ NEU                │
│ 3      ┆ 1:26 a cara do lucas kkkkkkkkk… ┆ NEU                │
│ 4      ┆ E é verdade que eu tô aqui      ┆ NEU                │
│ 5      ┆ Se vocês verem uma mota que é … ┆ NEU                │
│ 6      ┆ To-do is desenho tem so 4 dedo… ┆ NEG                │
│ 7      ┆ 847;63                          ┆ NEU                │
│ 8      ┆ 3.26                            ┆ NEU                │
│ 9      ┆ Não é a megue é a lisa          ┆ NEG                │
│ 10     ┆ Breet lembra um personagem de … ┆ NEU             

In [88]:
# Transformamos as colunas do Polars em listas do Python para o Scikit-Learn ler
y_verdadeiro = df_preenchido['Sentimentos_manual'].to_list()
y_predito = df_resultado['sentimento_predito'].to_list()

Visualizando se a quandidade de comentários não foi perdida, durante o processo

In [89]:
print(y_predito)
print(len(y_predito))

['POS', 'NEU', 'POS', 'NEU', 'NEU', 'NEU', 'NEU', 'NEU', 'NEU', 'NEU', 'NEU', 'POS', 'NEU', 'NEG', 'NEU', 'POS', 'NEG', 'POS', 'NEU', 'POS', 'NEU', 'POS', 'NEU', 'POS', 'POS', 'NEU', 'NEU', 'NEU', 'POS', 'POS', 'NEU', 'NEU', 'POS', 'NEU', 'POS', 'NEU', 'NEU', 'NEU', 'NEG', 'NEU', 'NEU', 'NEG', 'NEU', 'POS', 'POS', 'NEG', 'NEG', 'POS', 'NEG', 'NEU', 'NEG', 'NEU', 'NEU', 'NEU', 'POS', 'NEU', 'NEU', 'NEU', 'NEU', 'POS', 'NEU', 'NEU', 'NEU', 'NEU', 'POS', 'POS', 'POS', 'POS', 'POS', 'NEU', 'NEG', 'NEG', 'POS', 'POS', 'NEU', 'POS', 'POS', 'NEU', 'NEG', 'NEG', 'NEU', 'NEG', 'NEG', 'NEG', 'NEU', 'POS', 'NEU', 'NEG', 'NEU', 'NEU', 'NEU', 'POS', 'POS', 'NEU', 'POS', 'NEU', 'NEG', 'NEU', 'POS', 'NEU', 'NEG', 'NEG', 'NEU', 'NEG', 'POS', 'POS', 'POS', 'NEU', 'POS', 'NEU', 'NEU', 'POS', 'NEU', 'NEU', 'NEG', 'POS', 'POS', 'POS', 'NEG', 'NEU', 'POS', 'POS', 'POS', 'NEU', 'POS', 'NEU', 'NEG', 'NEU', 'NEU', 'NEU', 'NEU', 'NEU', 'NEG', 'NEU', 'NEG', 'POS', 'POS', 'POS', 'NEG', 'POS', 'NEU', 'NEU', 'NEU'

In [90]:
print(y_verdadeiro)
print(len(y_verdadeiro))

['POS', 'NEU', 'NEU', 'NEU', 'NEU', 'NEG', 'NEU', 'NEU', 'NEG', 'NEU', 'NEG', 'POS', 'NEU', 'NEG', 'NEU', 'POS', 'NEG', 'POS', 'NEG', 'POS', 'NEU', 'POS', 'NEG', 'POS', 'NEU', 'NEU', 'POS', 'NEU', 'POS', 'POS', 'NEU', 'NEG', 'POS', 'NEG', 'NEU', 'NEU', 'NEG', 'NEU', 'NEG', 'NEG', 'NEU', 'NEG', 'NEG', 'NEU', 'POS', 'NEG', 'NEG', 'POS', 'NEG', 'NEG', 'NEG', 'NEG', 'NEU', 'NEG', 'POS', 'NEG', 'NEU', 'NEG', 'NEU', 'POS', 'NEU', 'POS', 'NEG', 'NEU', 'NEU', 'NEU', 'POS', 'POS', 'POS', 'NEU', 'NEG', 'POS', 'NEU', 'POS', 'NEG', 'NEG', 'NEU', 'NEU', 'NEG', 'NEG', 'NEG', 'NEG', 'NEG', 'NEG', 'NEG', 'POS', 'NEG', 'NEG', 'NEG', 'NEG', 'NEU', 'POS', 'POS', 'NEG', 'POS', 'NEU', 'NEG', 'NEU', 'NEU', 'NEU', 'NEU', 'NEU', 'NEU', 'NEG', 'POS', 'POS', 'POS', 'NEU', 'NEU', 'POS', 'NEU', 'POS', 'NEG', 'POS', 'NEU', 'NEG', 'POS', 'POS', 'NEU', 'NEU', 'POS', 'POS', 'POS', 'NEU', 'POS', 'NEG', 'NEG', 'NEG', 'NEU', 'NEU', 'NEU', 'POS', 'POS', 'POS', 'NEG', 'POS', 'POS', 'POS', 'NEU', 'POS', 'NEU', 'NEU', 'NEG'

Calculando a perfomance do modelo 'pysentimento', verificando o acerto.

In [91]:
# Acurácia (Qual a porcentagem total de acertos?)
acuracia = accuracy_score(y_verdadeiro, y_predito)
print(f"🎯 Acurácia Geral do pysentimiento: {acuracia * 100:.2f}%")

# F1-Score (A média harmônica, excelente para ver se ele é bom em todas as classes)
# Usamos average='weighted' para balancear caso tenha muitos NEU e poucos POS
f1 = f1_score(y_verdadeiro, y_predito, average='weighted')
print(f"📊 F1-Score: {f1:.4f}")

# Matriz de Confusão (Onde o modelo está errando?)
labels = ["POS", "NEG", "NEU"]
matriz = confusion_matrix(y_verdadeiro, y_predito, labels=labels)

print("\n🔍 Matriz de Confusão:")
print(f"         Predito_POS | Predito_NEG | Predito_NEU")
print(f"Real_POS:    {matriz[0][0]:<7} | {matriz[0][1]:<7} | {matriz[0][2]}")
print(f"Real_NEG:    {matriz[1][0]:<7} | {matriz[1][1]:<7} | {matriz[1][2]}")
print(f"Real_NEU:    {matriz[2][0]:<7} | {matriz[2][1]:<7} | {matriz[2][2]}")

🎯 Acurácia Geral do pysentimiento: 56.85%
📊 F1-Score: 0.5525

🔍 Matriz de Confusão:
         Predito_POS | Predito_NEG | Predito_NEU
Real_POS:    152     | 11      | 44
Real_NEG:    27      | 46      | 96
Real_NEU:    58      | 38      | 163


## Sumário dos Resultados da Análise de Sentimento Calibrada

Após aplicar a função de calibragem de confiança, os resultados do modelo de análise de sentimento `pysentimiento` são os seguintes:

*   **Acurácia Geral:** 56.85%
    *   Este valor representa a proporção total de predições corretas (sentimentos positivos, negativos e neutros) em relação ao total de comentários analisados.

*   **F1-Score (Ponderado):** 0.5525
    *   O F1-Score é uma média harmônica entre a precisão e o recall, sendo uma métrica robusta, especialmente útil quando as classes estão desbalanceadas. Um F1-Score de 0.5525 indica um desempenho moderado do modelo em identificar corretamente os sentimentos em todas as classes, considerando a penalização aplicada aos neutros.

*   **Matriz de Confusão:**

|           | Predito_POS | Predito_NEG | Predito_NEU |
| :-------- | :---------- | :---------- | :---------- |
| **Real_POS**  | 152         | 11          | 44          |
| **Real_NEG**  | 27          | 46          | 96          |
| **Real_NEU**  | 58          | 38          | 163         |

### Interpretação da Matriz de Confusão:

*   **Verdadeiros Positivos (Real_POS | Predito_POS):** 152 comentários que eram realmente `POS` foram classificados corretamente como `POS`.
*   **Falsos Negativos (Real_POS | Predito_NEG / Predito_NEU):** Dos comentários `POS`, 11 foram classificados erroneamente como `NEG` e 44 como `NEU`.
*   **Verdadeiros Negativos (Real_NEG | Predito_NEG):** 46 comentários que eram realmente `NEG` foram classificados corretamente como `NEG`.
*   **Falsos Positivos (Real_NEG | Predito_POS / Predito_NEU):** Dos comentários `NEG`, 27 foram classificados erroneamente como `POS` e 96 como `NEU`.
*   **Verdadeiros Neutros (Real_NEU | Predito_NEU):** 163 comentários que eram realmente `NEU` foram classificados corretamente como `NEU`.
*   **Mal Classificados de Neutros (Real_NEU | Predito_POS / Predito_NEG):** Dos comentários `NEU`, 58 foram classificados erroneamente como `POS` e 38 como `NEG`.

Esta matriz oferece uma visão detalhada de onde o modelo acerta e erra, e como a calibragem afetou a classificação dos sentimentos, especialmente a propensão a classificar neutros como positivos ou negativos.

Utilizando uma penalização de 90% tivemos os resultados

In [ ]:
#🎯 Acurácia Geral do pysentimiento: 51.18%
#📊 F1-Score: 0.4838

#🔍 Matriz de Confusão:
#         Predito_POS | Predito_NEG | Predito_NEU
#Real_POS:    178     | 14      | 15
#Real_NEG:    53      | 72      | 44
#Real_NEU:    120     | 64      | 75

Utilizando uma penalização de 80%

In [ ]:
#🎯 Acurácia Geral do pysentimiento: 56.54%
#📊 F1-Score: 0.5526

#🔍 Matriz de Confusão:
#         Predito_POS | Predito_NEG | Predito_NEU
#Real_POS:    162     | 12      | 33
#Real_NEG:    35      | 57      | 77
#Real_NEU:    73      | 46      | 140